# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

# # Adjust this path if your repo is stored elsewhere in Drive.
# PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"

In [3]:
# # Install Python dependencies (run once per session)
# !pip install -r {PROJECT_ROOT}/requirements.txt -q
# !python -m spacy download en

In [4]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/COMP5329_Assignment1-main
!ls
import os
print(os.getcwd())

Mounted at /content/drive
/content/drive/MyDrive/COMP5329_Assignment1-main
assignment1.ipynb  Losses      requirements.txt			  Tools
_data		   _model      Schedulers			  TrainTools
Data		   Models      STAGE12_CODE_CHANGES_BY_MODULE.md
EvaluateTools	   Optimizers  STAGE1_DEBUG_LOG.md
_log		   README.md   STAGE2_DEBUG_LOG.md
/content/drive/MyDrive/COMP5329_Assignment1-main


In [5]:
#  Install Python dependencies (run once per session)
!pip install -r requirements.txt
!python -m spacy download en

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 2.8 MB/s eta 0:00:00
⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 154.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [6]:
import sys, os
# local root
from pathlib import Path
PROJECT_ROOT = str(Path.cwd().resolve())

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

Working directory: /content/drive/MyDrive/COMP5329_Assignment1-main


---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [7]:
from Tools.download import download_mini

download_mini(data_dir="_data")

Step 1 / 2  —  Mini dataset (SQuAD + GloVe)
  [skip] Mini dataset already present in _data/.

Step 2 / 2  —  spaCy language model
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

Mini dataset download complete.


---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [8]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

Generating train examples…


100%|██████████| 150/150 [00:02<00:00, 57.57it/s]


  30293 questions in total
Generating dev examples…


100%|██████████| 48/48 [00:00<00:00, 54.31it/s]


  10570 questions in total
Generating word embedding…


...

100%|██████████| 30293/30293 [00:02<00:00, 10769.23it/s]


  Built 30169 / 30293 instances
Processing dev examples…


100%|██████████| 10570/10570 [00:01<00:00, 9705.46it/s]


  Built 10465 / 10570 instances
Saving word embedding…
Saving char embedding…
Saving train eval…
Saving dev eval…
Saving word dictionary…
Saving char dictionary…
Saving dev meta…

Preprocessing complete.
  Outputs → _data/


{'train_record_file': '_data/train.npz',
 'dev_record_file': '_data/dev.npz',
 'word_emb_file': '_data/word_emb.json',
 'char_emb_file': '_data/char_emb.json',
 'train_eval_file': '_data/train_eval.json',
 'dev_eval_file': '_data/dev_eval.json',
 'word2idx_file': '_data/word2idx.json',
 'char2idx_file': '_data/char2idx.json',
 'dev_meta_file': '_data/dev_meta.json'}

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [9]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    num_steps  = 200,
    batch_size = 8,
    seed       = 42,

    # ── vanilla recipe: SGD, no scheduler, NLL loss ───────────────────────
    optimizer_name = "sgd",
    scheduler_name = "none",
    loss_name      = "qa_nll",
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

100%|██████████| 200/200 [00:13<00:00, 14.58it/s]


STEP      200  loss 1815.417674



100%|██████████| 150/150 [00:02<00:00, 62.35it/s]


VALID(train) loss 34.359869  F1 6.963300  EM 0.000000



100%|██████████| 150/150 [00:02<00:00, 64.49it/s]


TEST        loss 34.192778  F1 5.947087  EM 0.000000

Learning rate: [0.001]
Training finished.  Best F1: 5.9471  Best EM: 0.0000
Best F1: 5.9471  |  Best EM: 0.0000


Experiment: LayerNorm(A) vs GroupNorm(B)

Experiment A

In [15]:
from TrainTools.train import train

results_layer_norm = train(
    # data paths
    train_npz        = "_data/train.npz",
    dev_npz          = "_data/dev.npz",
    word_emb_json    = "_data/word_emb.json",
    char_emb_json    = "_data/char_emb.json",
    train_eval_json  = "_data/train_eval.json",
    dev_eval_json    = "_data/dev_eval.json",
    save_dir         = "_model_layer_norm",
    log_dir          = "_log_layer_norm",
    ckpt_name        = "model.pt",

    # training loop
    batch_size       = 8,
    num_steps        = 400,
    checkpoint       = 200,
    val_num_batches  = 150,
    test_num_batches = 150,
    seed             = 42,

    # optimization
    optimizer_name   = "sgd",
    scheduler_name   = "none",
    loss_name        = "qa_nll",

    # normalization experiment
    norm_name        = "layer_norm",
    norm_groups      = 8,
)

print(f"LayerNorm | Best F1: {results_layer_norm['best_f1']:.4f} | Best EM: {results_layer_norm['best_em']:.4f}")

100%|██████████| 200/200 [00:13<00:00, 14.45it/s]


STEP      200  loss 1815.417674



100%|██████████| 150/150 [00:02<00:00, 65.87it/s]


VALID(train) loss 34.359869  F1 6.963300  EM 0.000000



100%|██████████| 150/150 [00:02<00:00, 64.92it/s]


...

STEP      400  loss 1056.490427



100%|██████████| 150/150 [00:02<00:00, 65.84it/s]


VALID(train) loss 32.915061  F1 6.956789  EM 0.000000



100%|██████████| 150/150 [00:02<00:00, 65.96it/s]


TEST        loss 32.296999  F1 6.143611  EM 0.000000

Learning rate: [0.001]
Training finished.  Best F1: 6.1436  Best EM: 0.0000
LayerNorm | Best F1: 6.1436 | Best EM: 0.0000


Experiment B

In [16]:
from TrainTools.train import train

results_group_norm = train(
    # data paths
    train_npz        = "_data/train.npz",
    dev_npz          = "_data/dev.npz",
    word_emb_json    = "_data/word_emb.json",
    char_emb_json    = "_data/char_emb.json",
    train_eval_json  = "_data/train_eval.json",
    dev_eval_json    = "_data/dev_eval.json",
    save_dir         = "_model_group_norm",
    log_dir          = "_log_group_norm",
    ckpt_name        = "model.pt",

    # training loop
    batch_size       = 8,
    num_steps        = 400,
    checkpoint       = 200,
    val_num_batches  = 150,
    test_num_batches = 150,
    seed             = 42,

    # optimization
    optimizer_name   = "sgd",
    scheduler_name   = "none",
    loss_name        = "qa_nll",

    # normalization experiment
    norm_name        = "group_norm",
    norm_groups      = 8,
)

print(f"GroupNorm | Best F1: {results_group_norm['best_f1']:.4f} | Best EM: {results_group_norm['best_em']:.4f}")

100%|██████████| 200/200 [00:14<00:00, 13.66it/s]


STEP      200  loss 2318.201827



100%|██████████| 150/150 [00:02<00:00, 50.45it/s]


VALID(train) loss 40.583147  F1 6.554975  EM 0.083333



100%|██████████| 150/150 [00:02<00:00, 50.83it/s]


...

STEP      400  loss 1309.767938



100%|██████████| 150/150 [00:02<00:00, 51.13it/s]


VALID(train) loss 34.641206  F1 5.886256  EM 0.083333



100%|██████████| 150/150 [00:02<00:00, 51.20it/s]


TEST        loss 35.040361  F1 5.629495  EM 0.083333

Learning rate: [0.001]
Training finished.  Best F1: 5.6295  Best EM: 0.0833
GroupNorm | Best F1: 5.6295 | Best EM: 0.0833


In [17]:
print("===== Training Summary =====")
print(f"LayerNorm | Best F1: {results_layer_norm['best_f1']:.4f} | Best EM: {results_layer_norm['best_em']:.4f}")
print(f"GroupNorm | Best F1: {results_group_norm['best_f1']:.4f} | Best EM: {results_group_norm['best_em']:.4f}")

===== Training Summary =====
LayerNorm | Best F1: 6.1436 | Best EM: 0.0000
GroupNorm | Best F1: 5.6295 | Best EM: 0.0833


---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [11]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

100%|██████████| 1309/1309 [00:20<00:00, 65.23it/s]


TEST  loss 34.829248  F1 8.061374  EM 0.019111
F1: 8.0614  |  EM: 0.0191  |  Loss: 34.829248


Custom evaluation function

In [23]:
import os
import argparse
import torch
import ujson as json

from Data import SQuADDataset, load_dev_eval, load_word_char_mats
from Losses import losses
from Models import QANet
from EvaluateTools.eval_utils import run_eval

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def evaluate_norm(
    dev_npz="_data/dev.npz",
    word_emb_json="_data/word_emb.json",
    char_emb_json="_data/char_emb.json",
    dev_eval_json="_data/dev_eval.json",
    save_dir="_model",
    log_dir="_log",
    ckpt_name="model.pt",
    batch_size=8,
    test_num_batches=-1,
    loss_name="qa_nll",
    para_limit=400,
    ques_limit=50,
    char_limit=16,
    d_model=96,
    num_heads=8,
    glove_dim=300,
    char_dim=64,
    dropout=0.1,
    dropout_char=0.05,
    pretrained_char=False,
    norm_name="layer_norm",
    norm_groups=8,
    activation="relu",
    init_name="kaiming",
):
    os.makedirs(log_dir, exist_ok=True)

    args = argparse.Namespace(
        dev_npz=dev_npz,
        word_emb_json=word_emb_json,
        char_emb_json=char_emb_json,
        dev_eval_json=dev_eval_json,
        para_limit=para_limit,
        ques_limit=ques_limit,
        char_limit=char_limit,
        d_model=d_model,
        num_heads=num_heads,
        glove_dim=glove_dim,
        char_dim=char_dim,
        dropout=dropout,
        dropout_char=dropout_char,
        pretrained_char=pretrained_char,
        norm_name=norm_name,
        norm_groups=norm_groups,
        activation=activation,
        init_name=init_name,
    )

    word_mat, char_mat = load_word_char_mats(args)
    model = QANet(word_mat, char_mat, args).to(DEVICE)

    dev_eval = load_dev_eval(args)
    dev_dataset = SQuADDataset(dev_npz)

    ckpt_path = os.path.join(save_dir, ckpt_name)
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state"])

    metrics, ans = run_eval(
        model,
        dev_dataset,
        dev_eval,
        num_batches=test_num_batches,
        batch_size=batch_size,
        use_random_batches=False,
        device=DEVICE,
        loss_fn=losses[loss_name],
    )

    with open(os.path.join(log_dir, "answers.json"), "w") as f:
        json.dump(ans, f)

    print("TEST  loss {loss:.6f}  F1 {f1:.6f}  EM {exact_match:.6f}".format(**metrics))
    return {
        "f1": metrics["f1"],
        "exact_match": metrics["exact_match"],
        "loss": metrics["loss"],
    }

Evaluate A

In [24]:
metrics_layer_norm = evaluate_norm(
    dev_npz        = "_data/dev.npz",
    word_emb_json  = "_data/word_emb.json",
    char_emb_json  = "_data/char_emb.json",
    dev_eval_json  = "_data/dev_eval.json",
    save_dir       = "_model_layer_norm",
    log_dir        = "_log_layer_norm",
    ckpt_name      = "model.pt",
    norm_name      = "layer_norm",
    norm_groups    = 8,
)

print(f"LayerNorm | F1: {metrics_layer_norm['f1']:.4f} | EM: {metrics_layer_norm['exact_match']:.4f} | Loss: {metrics_layer_norm['loss']:.4f}")

100%|██████████| 1309/1309 [00:20<00:00, 65.05it/s]


TEST  loss 33.156413  F1 8.275679  EM 0.047778
LayerNorm | F1: 8.2757 | EM: 0.0478 | Loss: 33.1564


Evaluate B

In [25]:
metrics_group_norm = evaluate_norm(
    dev_npz        = "_data/dev.npz",
    word_emb_json  = "_data/word_emb.json",
    char_emb_json  = "_data/char_emb.json",
    dev_eval_json  = "_data/dev_eval.json",
    save_dir       = "_model_group_norm",
    log_dir        = "_log_group_norm",
    ckpt_name      = "model.pt",
    norm_name      = "group_norm",
    norm_groups    = 8,
)

print(f"GroupNorm | F1: {metrics_group_norm['f1']:.4f} | EM: {metrics_group_norm['exact_match']:.4f} | Loss: {metrics_group_norm['loss']:.4f}")

100%|██████████| 1309/1309 [00:25<00:00, 51.03it/s]


TEST  loss 36.012694  F1 6.863290  EM 0.114668
GroupNorm | F1: 6.8633 | EM: 0.1147 | Loss: 36.0127


In [26]:
print("===== Final Comparison =====")
print(f"LayerNorm | Train Best F1: {results_layer_norm['best_f1']:.4f} | Train Best EM: {results_layer_norm['best_em']:.4f} | Eval F1: {metrics_layer_norm['f1']:.4f} | Eval EM: {metrics_layer_norm['exact_match']:.4f}")
print(f"GroupNorm | Train Best F1: {results_group_norm['best_f1']:.4f} | Train Best EM: {results_group_norm['best_em']:.4f} | Eval F1: {metrics_group_norm['f1']:.4f} | Eval EM: {metrics_group_norm['exact_match']:.4f}")

===== Final Comparison =====
LayerNorm | Train Best F1: 6.1436 | Train Best EM: 0.0000 | Eval F1: 8.2757 | Eval EM: 0.0478
GroupNorm | Train Best F1: 5.6295 | Train Best EM: 0.0833 | Eval F1: 6.8633 | Eval EM: 0.1147


Experiment: Effect of Normalization Strategy on QANet

Research Question
How does the choice of normalization method (LayerNorm vs GroupNorm) affect model performance in the repaired QANet?

Hypothesis
We hypothesize that LayerNorm will outperform GroupNorm in this task, as LayerNorm is better suited for sequence-based models and is commonly used in transformer-style architectures.

Experimental Setup
We conduct a controlled experiment by modifying only the normalization strategy while keeping all other parameters fixed.

Model: repaired QANet
Dataset: SQuAD v1.1
Batch size: 8
Training steps: 400
Optimizer: SGD
Scheduler: none
Loss: QA NLL
Seed: 42

Two configurations were evaluated:
LayerNorm
GroupNorm (8 groups)

Results
Setting	Train F1	Train EM	Eval F1	Eval EM
LayerNorm	6.1436	0.0000	8.2757	0.0478
GroupNorm	5.6295	0.0833	6.8633	0.1147

Analysis
The results show that LayerNorm significantly outperforms GroupNorm in terms of F1 score, indicating better overall answer quality. This suggests that LayerNorm is more suitable for the QANet architecture in this task.

Interestingly, GroupNorm achieves a higher Exact Match (EM), meaning it produces more perfectly correct answers. However, its lower F1 score indicates that its predictions are generally less accurate in terms of answer overlap.

One possible explanation is that LayerNorm operates across feature dimensions within each token, making it more compatible with sequence modeling and attention mechanisms. In contrast, GroupNorm normalizes across channel groups, which may not align well with the structure of the QANet model.

As a result, LayerNorm provides more stable and consistent predictions, while GroupNorm occasionally produces exact matches but with lower overall quality.

Limitations and Future Work
This experiment was conducted with a relatively small number of training steps (400), which may limit the model's convergence. Future work could include longer training or multiple random seeds to obtain more robust conclusions.